In [1]:
!pip install transformers peft trl datasets bitsandbytes accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 10.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("hf_token")

login(token=hf_token)

In [3]:
from transformers import pipeline

question = "x^2 + 5x - 6 = 0, what is x?"
generator = pipeline("text-generation", model="AryanK123/Llama-3.2-1B-Instruct_SFT_Math-220kv00.04", device="cuda")
output = generator([{"role": "user", "content": question}], max_new_tokens=4096, return_full_text=False)[0]
print(output["generated_text"])

config.json:   0%|          | 0.00/858 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Okay, so I need to solve the quadratic equation x² + 5x - 6 = 0. Hmm, let's see. I remember there are a couple of ways to solve quadratic equations: factoring, completing the square, and the quadratic formula. Let me try factoring first because if it works, that might be the quickest method. 

First, I need to find two numbers that multiply to -6 (the constant term) and add up to 5 (the coefficient of the x term). Let me think... The factors of -6 could be 2 and -3 because 2 * (-3) = -6 and 2 + (-3) = -1, but that's not right. Wait, maybe I should list all pairs of factors of 6: (1,6), (2,3), (-1,-6), (-2,-3). Hmm, none of those pairs add up to 5. So, that means the equation doesn't factor nicely over integers. So maybe I need to use the quadratic formula then?

The quadratic formula is x = [-b ± √(b² - 4ac)] / (2a). Let me identify a, b, and c from the equation. The equation is x² + 5x - 6 = 0, so comparing to ax² + bx + c, we have a = 1, b = 5, and c = -6. Plugging these into

In [11]:
import os
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = "AryanK123/Llama-3.2-1B-Instruct_SFT_Math-220kv00.04"
HUB_TARGET_REPO = "AryanK123/Llama-3.2-1B-GRPO-DeepMath"
DATASET_ID = "zwhe99/DeepMath-103K"
TOTAL_SAMPLES = 7000

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print("Loading raw dataset...")
raw_dataset = load_dataset(DATASET_ID, split="train")

metadata_df = pd.DataFrame({
    "idx": np.arange(len(raw_dataset)),
    "strata": np.round(raw_dataset["difficulty"]).astype(int)
})

BUFFER_SAMPLES = int(TOTAL_SAMPLES * 1.22)
total_available = len(metadata_df)
strata_counts = metadata_df["strata"].value_counts()

sampled_indices = []
for strata_val, count in strata_counts.items():
    subset = metadata_df[metadata_df["strata"] == strata_val]
    n_alloc = max(1, int(round((count / total_available) * BUFFER_SAMPLES)))
    sampled_indices.extend(subset.sample(n=min(n_alloc, len(subset)), random_state=42)["idx"].tolist())

sampled_dataset = raw_dataset.select(sampled_indices)

def process_and_select_trace(example):
    """
    Evaluates r1_solution_1, 2, 3 and selects the trace with the 
    minimum token count that is strictly <= 4096 tokens.
    """
    candidate_solutions = [
        example.get("r1_solution_1"),
        example.get("r1_solution_2"),
        example.get("r1_solution_3"),
    ]
    
    valid_traces = []
    for sol in candidate_solutions:
        if sol and isinstance(sol, str) and sol.strip():
            if len(sol) > 25000:
                continue
            token_count = len(tokenizer(sol, truncation=False, add_special_tokens=False)["input_ids"])
            if token_count <= 4096:
                valid_traces.append((sol, token_count))
    
    if not valid_traces:
        return {"has_valid_trace": False, "selected_trace": None}
    
    best_trace, _ = min(valid_traces, key=lambda x: x[1])
    return {"has_valid_trace": True, "selected_trace": best_trace}

# Map and filter ONLY the sampled slice (~8.5k rows)
processed_sample = sampled_dataset.map(process_and_select_trace, num_proc=4)
valid_sample = processed_sample.filter(lambda x: x["has_valid_trace"], num_proc=4)

# Trim to exact target count
final_sample = valid_sample.select(range(min(TOTAL_SAMPLES, len(valid_sample))))

def format_prompt(example):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a mathematical reasoning assistant. First, think through your steps "
                "carefully between <think> and </think> tags. Then, provide your final response "
                "and enclose the final answer inside \\boxed{}."
            )
        },
        {"role": "user", "content": example["question"]}
    ]
    return {
        "prompt": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "target_answer": str(example["final_answer"]).strip(),
        "baseline_trace": str(example["selected_trace"]).strip(),
        "difficulty": float(example["difficulty"])
    }

grpo_dataset = final_sample.map(
    format_prompt,
    remove_columns=final_sample.column_names,
    num_proc=4
)
print(f"Dataset prepared with {len(grpo_dataset)} samples for GRPO.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0}
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

def accuracy_reward_func(completions, target_answer, **kwargs):
    rewards = []
    for completion, target in zip(completions, target_answer):
        matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", completion)
        extracted = matches[-1].strip() if matches else ""

        target_clean = target.strip()
        target_matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", target_clean)
        if target_matches:
            target_clean = target_matches[-1].strip()

        rewards.append(2.0 if extracted and extracted == target_clean else 0.0)
    return rewards

def format_tag_reward_func(completions, **kwargs):
    rewards = []
    strict_pattern = re.compile(r"^<think>(.*?)</think>(.+)$", re.DOTALL)
    for comp in completions:
        match = strict_pattern.match(comp.strip())
        rewards.append(0.5 if match and len(match.group(1).strip()) > 20 else 0.0)
    return rewards

def boxed_presence_reward_func(completions, **kwargs):
    rewards = []
    for comp in completions:
        rewards.append(0.3 if "\\boxed{" in comp else 0.0)
    return rewards

training_args = GRPOConfig(
    output_dir="./llama-3.2-1b-grpo-deepmath",
    learning_rate=2e-5,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=5,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=4096,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,

    # vLLM Integration (vllm_device removed)
    use_vllm=False,
    # Hugging Face Hub Auto-Saving
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    push_to_hub=True,
    hub_model_id=HUB_TARGET_REPO,
    hub_strategy="checkpoint",
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    peft_config=peft_config,
    reward_funcs=[accuracy_reward_func, format_tag_reward_func, boxed_presence_reward_func],
    args=training_args,
    train_dataset=grpo_dataset,
)


trainer.train()
trainer.save_model("./final-checkpoint")
trainer.push_to_hub(commit_message="Training complete - Final GRPO Model")
tokenizer.push_to_hub(HUB_TARGET_REPO)

Loading raw dataset...
Dataset prepared with 5246 samples for GRPO.


Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


KeyboardInterrupt: 

In [1]:
%%bash
git status

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


CalledProcessError: Command 'b'git status\n'' returned non-zero exit status 128.